## Prompts In Langchain ##

**Static Prompts->You directly pass the string prompt to invoke the whole control is with user.**

**Dyanamic Prompts->You define a template and take values from user and put in it.**

**We can also use the f string in variables promts,but the prompttemplate we have default validation,we can save the prompt template as json and later can load also it is fit with langchain ecosystem.**

In [3]:
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint

import os
from dotenv import load_dotenv
load_dotenv()

hug_token = os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
llm = HuggingFaceEndpoint(repo_id="Qwen/Qwen2.5-7B-Instruct",task="text-generation",huggingfacehub_api_token=hug_token)

model = ChatHuggingFace(llm=llm)



In [11]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    template = """

    You are an experienced research explainer. Explain the given paper: "{paper_input}", the style must be "{style_input}", and length needs to be "{length_input}",words.
    If you do not have enough information say "I don't have enough information.", instead of guessing.
    """,
    input_variables = [
        "paper_input",
        "style_input",
        "length_input", ]
)

paper_input = "word2vec"
style_input = "summary"
length_input = "8-paragraphs"

chain = template | model

result = chain.invoke({
        "paper_input": paper_input,
        "style_input": style_input,
        "length_input": length_input,
})

print(result.content)


I don't have enough information.

The paper "word2vec" is not a single published paper but a collective term for a family of related models used for learning word representations in the field of natural language processing (NLP). The term encompasses two main models: Continuous Bag-of-Words (CBOW) and Skip-gram. These models are part of a broader effort to represent words in a numerical format that captures their context and meaning, making it possible to apply techniques from fields like machine learning and deep learning to NLP tasks.

word2vec models are designed to capture the semantic and syntactic properties of words by learning dense vector representations that can be used as inputs to other machine learning models. The primary advantage of these models is their efficiency and ability to handle large-scale datasets. They achieve this by modeling the context of a word, which is the set of words that appear around it, to predict the target word or vice versa.

CBOW predicts the ta

**messages->use to keep history.sytem messge(it is a role or we can say like you are a doctor etc.),Humanmessage(User,s prompt) and AiMessage**

In [15]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage

chat_history = [SystemMessage(content="You are a helpful AI assistant.")]

while True:
    user_input = input("You: ")
    if user_input == "exit":
        break
    chat_history.append(HumanMessage(content=user_input))
    result = model.invoke(chat_history)
    chat_history.append(AIMessage(content=result.content))
    print(result.content)

print(chat_history)

Hello! How can I assist you today?
The number 2 is greater than the number 1.
The greater number is 2. When you multiply 2 by 10, you get:

\[ 2 \times 10 = 20 \]
[SystemMessage(content='You are a helpful AI assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Which number is greater 2 or 1', additional_kwargs={}, response_metadata={}), AIMessage(content='The number 2 is greater than the number 1.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='multiply greater number ith 10', additional_kwargs={}, response_metadata={}), AIMessage(content='The greater number is 2. When you multiply 2 by 10, you get:\n\n\\[ 2 \\times 10 = 20 \\]', additional_kwargs={}, response_metadata={}, tool_calls=[], invali

**prompttemplate -> for static  messages or prompts. ChatPrompttemplate-> For dynamic list of messages(we can create a placeholder).**

In [6]:
#chatprompttemplate ->little bit different than the prompttemplate
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate([
    ("system","You are a {domain} expert."),
    ("human","explain the {topic}")]
)

prompt = chat_template.invoke({"domain":"cricket","topic":"run out"})
print(prompt)


messages=[SystemMessage(content='You are a cricket expert.', additional_kwargs={}, response_metadata={}), HumanMessage(content='explain the run out', additional_kwargs={}, response_metadata={})]


In [ ]:
#placeholder where we can add or chats to a file and retrive for context

from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder
chat_history = []

chat_template = ChatPromptTemplate([
    ("system","You are a {domain} expert."),
    MessagesPlaceholder(variable_name='chat_history'),
    ("human","explain the {topic}")
])

with open('chat_history.txt') as f:
    chat_history.extend(f.readlines())

prompt = chat_template.invoke({"chat_history":chat_history,"query":"How are you."})

